In [51]:
import numpy as np
# -----------------------------------------
# STEP 1: Define the Hidden States (Tags)
# -----------------------------------------
tags = ["Article","Noun", "Verb","Adverb"]

In [52]:
# -----------------------------------------
# STEP 2: Define the Vocabulary
# -----------------------------------------
words = ["A", "car", "is", "moving", "straight"]

# Convert words into indices
word_to_index = {
"A": 0,
"car": 1,
"is": 2,
"moving": 3,
"straight": 4,
}

In [53]:
# -----------------------------------------
# STEP 3: Initial Probability
# P(Tag at first word)
# -----------------------------------------
init_prob = np.array([
 0.6,  # Article
0.2,  # Noun
0.1,  # Verb
0.1,  # Adverb
])

In [54]:
# -----------------------------------------
# STEP 4: Transition Probability Matrix
# P(Current Tag | Previous Tag)
#
# (fixed: this was a 2x2 matrix but there are 4 tags, so it needs to be 4x4.
# Row = From tag, Column = To tag, in the order [Article, Noun, Verb, Adverb])
#
#              To:  Article  Noun  Verb  Adverb
# From Article        0.05  0.85  0.05    0.05
# From Noun            0.05  0.10  0.70    0.15
# From Verb            0.10  0.20  0.10    0.60
# From Adverb          0.10  0.30  0.50    0.10
# -----------------------------------------
transition = np.array([
    [0.05, 0.85, 0.05, 0.05],  # From Article
    [0.05, 0.10, 0.70, 0.15],  # From Noun
    [0.10, 0.20, 0.10, 0.60],  # From Verb
    [0.10, 0.30, 0.50, 0.10],  # From Adverb
])

In [55]:
# -----------------------------------------
# STEP 5: Emission Probability Matrix
# P(Word | Tag)
#
# (fixed: this was a 2x2 matrix for unrelated words "boy"/"crying", but the
# vocabulary actually has 5 words and there are 4 tags, so it needs to be
# a 4x5 matrix. Row = Tag, Column = Word, in the order
# [A, car, is, moving, straight])
#
#              A     car    is    moving  straight
# Article    0.900  0.025  0.025  0.025   0.025
# Noun       0.020  0.850  0.020  0.080   0.030
# Verb       0.020  0.030  0.550  0.350   0.050
# Adverb     0.020  0.030  0.050  0.100   0.800
# -----------------------------------------
emission = np.array([
    [0.900, 0.025, 0.025, 0.025, 0.025],  # Article
    [0.020, 0.850, 0.020, 0.080, 0.030],  # Noun
    [0.020, 0.030, 0.550, 0.350, 0.050],  # Verb
    [0.020, 0.030, 0.050, 0.100, 0.800],  # Adverb
])

In [56]:
# -----------------------------------------
# STEP 6: Convert probabilities into Log Space
# This avoids numerical underflow.
# -----------------------------------------
log_init = np.log(init_prob + 1e-10)
log_trans = np.log(transition + 1e-10)
log_emit = np.log(emission + 1e-10)

In [57]:
import numpy as np

# -----------------------------------------
# STEP 7: Viterbi Algorithm
# -----------------------------------------
def viterbi(sentence):
    # Number of words
    T = len(sentence)

    # Number of tags
    N = len(tags)

    # DP table
    dp = np.full((N, T), -np.inf)

    # Backpointer table
    backpointer = np.zeros((N, T), dtype=int)

    # Initialization
    dp[:, 0] = log_init + log_emit[:, sentence[0]]

    # Recursion
    for t in range(1, T):
        for j in range(N):
            scores = (
                dp[:, t - 1]
                + log_trans[:, j]
                + log_emit[j, sentence[t]]
            )

            backpointer[j, t] = np.argmax(scores)
            dp[j, t] = np.max(scores)

    # Backtracking
    best_path = [np.argmax(dp[:, -1])]

    for t in range(T - 1, 0, -1):
        best_path.append(backpointer[best_path[-1], t])

    best_path.reverse()

    return best_path

In [58]:
# -----------------------------------------
# STEP 8: Example Sentence
# -----------------------------------------
sentence = ["A", "car", "is", "moving", "straight"]

sentence_index = []

for word in sentence:
    if word in word_to_index:
        sentence_index.append(word_to_index[word])
    else:
        print(f"Word '{word}' not found in vocabulary.")
        print("Available words:", list(word_to_index.keys()))
        break
else:
    best_tags = viterbi(sentence_index)

In [59]:
# -----------------------------------------
# STEP 9: Print Results
# -----------------------------------------

print("Sentence:", sentence)

print("\nPredicted Tags:")
for word, tag in zip(sentence, best_tags):
    print(word, "->", tags[tag])

Sentence: ['A', 'car', 'is', 'moving', 'straight']

Predicted Tags:
A -> Article
car -> Noun
is -> Verb
moving -> Verb
straight -> Adverb
